# Certilab Adaptive RAG — Demo

Implementación del patrón Adaptive RAG con LangGraph + OpenAI + Tavily.
7 nodos, 2 loops de auto-corrección.

**Artículo**: [Building an Adaptive RAG System](https://levelup.gitconnected.com/building-an-adaptive-rag-system-with-langgraph-openai-and-tavily-c4ee39d2f021)

💡 **Colab**: ejecutá la celda 1 para clonar. **Local**: salteala.


In [ ]:
import os, sys
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/emersonheto/certilab-adaptive-rag.git /content/certilab-adaptive-rag
    %cd /content/certilab-adaptive-rag
    !pip install -q langgraph langchain-openai langchain-core openai tavily-python pydantic pydantic-settings python-dotenv
    print('✅ Repo clonado')
else:
    print('ℹ️  Local')


In [ ]:
if IN_COLAB:
    try:
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')
        print('✅ Colab Secrets')
    except Exception:
        from dotenv import load_dotenv
        from pathlib import Path
        env = Path.cwd() / '.env'
        if env.exists():
            load_dotenv(env, override=False)
            print('✅ .env')
        else:
            os.environ['OPENAI_API_KEY'] = input('🔑 OPENAI_API_KEY: ').strip()
            k = input('🔑 TAVILY_API_KEY (Enter=skip): ').strip()
            if k:
                os.environ['TAVILY_API_KEY'] = k
else:
    from dotenv import load_dotenv
    from pathlib import Path
    env = Path.cwd() / '.env'
    load_dotenv(env if env.exists() else None, override=False)

if not os.getenv('OPENAI_API_KEY'):
    print('❌ OPENAI_API_KEY faltante')
    raise RuntimeError('OPENAI_API_KEY faltante')
has_tavily = bool(os.getenv('TAVILY_API_KEY'))
print(f'✅ OPENAI | TAVILY: {chr(9989) if has_tavily else chr(10060)}')


In [ ]:
from app.adaptive_rag.graph import build_graph
from app.adaptive_rag.state import AdaptiveRAGState
from app.config import Settings
from app.domain.models import Role
from app.security.access_control import Principal, scope_from_principal
from app.tools.web_search import TavilyWebSearch, WebSearchConfig
print('✅ Imports')


In [ ]:
from pathlib import Path
settings = Settings()
print(f'Modo: {settings.app_mode}')

if settings.app_mode != 'real':
    from app.ingestion.loader import load_certificates, load_customers, load_histories, load_pdf_texts
    from app.ingestion.splitter import build_pdf_chunks
    from app.ingestion.indexer import InMemoryVectorIndex
    data_dir = Path('data')
    certs = load_certificates(data_dir)
    load_customers(data_dir)
    load_histories(data_dir)
    chunks = build_pdf_chunks(certs, load_pdf_texts(data_dir))
    index = InMemoryVectorIndex(chunks=chunks)
    embedding_provider = None
    print(f'✅ Mock: {len(certs)} certificados')
else:
    from app.retrieval.qdrant_index import QdrantVectorIndex
    from app.tools.embeddings import EmbeddingProviderConfig, EmbeddingsProvider
    embedding_provider = EmbeddingsProvider(EmbeddingProviderConfig.from_settings(settings))
    index = QdrantVectorIndex.from_settings(settings, embedding_provider)
    print(f'✅ Qdrant | {settings.openai_embedding_model}')

tavily_key = settings.tavily_api_key
web_search = TavilyWebSearch(WebSearchConfig(tavily_api_key=tavily_key))
graph = build_graph(index=index, embeddings=embedding_provider, web_search=web_search, settings=settings)
print(f'✅ Grafo compilado')


In [ ]:
def make_state(q):
    p = Principal(role=Role.ADMIN, customer_id=None, user_id=1)
    s = scope_from_principal(p)
    return {'question': q, 'generation': '', 'documents': [], 'web_results': [],
            'route': '', 'rewrite_count': 0, 'regenerate_count': 0,
            'hallucination_verdict': '', 'answer_verdict': '', 'principal': p, 'scope': s}

def run(q):
    state = make_state(q)
    route = rewrites = regenerates = 0
    answer = grounded = useful = '?'
    for step in graph.stream(state):
        for _, out in step.items():
            if isinstance(out, dict):
                if 'route' in out:
                    route = out['route']
                if 'rewrite_count' in out:
                    rewrites = out['rewrite_count']
                if 'regenerate_count' in out:
                    regenerates = out['regenerate_count']
                if 'generation' in out:
                    answer = out['generation']
                if 'hallucination_verdict' in out:
                    grounded = out['hallucination_verdict']
                if 'answer_verdict' in out:
                    useful = out['answer_verdict']
    print(f'Ruta: {route}')
    if rewrites:
        print(f'Reescrituras: {rewrites}')
    if regenerates:
        print(f'Regeneraciones: {regenerates}')
    print(f'Verificado: {grounded} | Util: {useful}')
    print(f'\n{answer}')
print('✅ Helpers')


## Query A: Procedimiento INDECOPI


In [ ]:
run("¿Qué procedimiento de calibración sigue la norma INDECOPI?")


## Query B: Tenant isolation — ALS PERU


In [ ]:
run("¿Qué certificados tiene ALS PERU?")


## Self-Correction Loops

1. **Rewrite**: docs irrelevantes → reescribe → reintenta (max 3)
2. **Regenerate**: alucina → regenera (max 2)


### Loop 1: Reescritura


In [ ]:
run("Dame info de calibración")


### Query D: Datos de tablas


In [ ]:
run("¿Cuál fue la temperatura máxima a 105°C?")


### Query E: Metadatos


In [ ]:
run("¿Qué certificados acreditados se emitieron en mayo 2026?")


## Visualización


In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


## Resumen

- Enrutamiento adaptativo
- Tenant isolation
- Gradeo de relevancia
- Reescritura de query (max 3)
- Verificacion de alucinaciones (max 2)
- 154 certificados reales
- Pipeline: PyMuPDF + Camelot + Unstructured -> Qdrant
